# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Configuring base model: {MODEL_ID}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit Quantization configuration for GPU execution (e.g. Google Colab T4)
# with automatic fallback to CPU if CUDA is not available in local environment
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    print("CUDA detected: Loading model with 4-bit quantization (~750MB VRAM)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    print("CUDA not detected: Falling back to CPU execution...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cpu",
        dtype=torch.bfloat16 if hasattr(torch, 'bfloat16') else torch.float32,
        trust_remote_code=True
    )

print("Base model loaded successfully!")


Configuring base model: Qwen/Qwen2.5-1.5B-Instruct


CUDA not detected: Falling back to CPU execution...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded successfully!


In [2]:
# Ambiguous customer inquiry regarding a delayed order (no policy context provided)
test_query = "My order hasn't arrived yet and it's been several days. When will it get here and can I get a refund?"

messages = [
    {"role": "system", "content": "You are a customer support agent. Answer the user's inquiry helpfully and accurately."},
    {"role": "user", "content": test_query}
]

# Format prompt using model's chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Test Query:\n\"{test_query}\"\n")
print("Generating baseline response (zero-shot, no retrieval context)...")

# Deterministic generation settings per guidelines:
# do_sample=False, temperature=None, top_p=None to avoid deprecation warnings
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id
    )

# Extract only newly generated tokens
generated_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
baseline_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- Baseline Model Output ---")
print(baseline_output)


Test Query:
"My order hasn't arrived yet and it's been several days. When will it get here and can I get a refund?"

Generating baseline response (zero-shot, no retrieval context)...



--- Baseline Model Output ---
I'm sorry to hear that your order has not arrived as expected. Please provide me with your order number so I can assist you further. As for when your order might arrive, please note that delivery times can vary depending on your location and shipping method chosen by the seller.

Regarding refunds, if there is an issue with your order, we recommend contacting the seller directly through their website or app. They should be able to guide you on how to initiate a return or exchange process. If they do not offer this service, you may need to contact the seller's customer service department for assistance. It's


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
# Ground truth policy rule from corporate_policies/shipping_delays.md:
ground_truth = (
    "Domestic orders deliver within 3-7 business days; international orders within 10-21 business days. "
    "An order is only considered 'delayed' once it has passed the upper bound of its quoted window. "
    "Carrier scans can lag reality by up to 24 hours, so a single stale tracking event is not by itself a delay. "
    "If the parcel is late but in transit, open a carrier trace with a 24-48 hour follow-up. "
    "If no movement for >5 business days, treat as lost and offer replacement or refund."
)

print("=== Factual Accuracy Assessment against Corporate SOP ===")
print(f"\n[Ground-Truth Policy (shipping_delays.md)]:\n{ground_truth}\n")
print(f"[Baseline Model Output]:\n{baseline_output}\n")

# Hallucination and factual gap analysis:
hallucinations = []

# 1. Timeline checks
hallucinations.append(
    "Factual Inaccuracy / Timeline Hallucination: The baseline model does not know the company's 3-7 business day domestic "
    "or 10-21 day international standard. It either invents generic delivery windows or asks the user to check third-party estimates."
)

# 2. SOP Procedure checks
hallucinations.append(
    "Procedural Gap / Missing Protocol: The baseline fails to mention the mandatory carrier trace protocol (expect update in 24-48 hours) "
    "or the 5 business days with no movement rule required before qualifying for a lost-package refund/replacement."
)

# 3. Policy Hallucination
hallucinations.append(
    "Unverified Commitment Risk: The baseline model may suggest contacting support for an immediate refund without verifying if the quoted "
    "delivery window has passed, violating company policy that orders are only delayed after the upper delivery window bound."
)

print("--- Identified Hallucinations & Limitations ---")
for idx, h in enumerate(hallucinations, 1):
    print(f"{idx}. {h}")

print("\n--- Learner Inference ---")
print(
    "Inference: The base LLM produces fluent, polite text, but lacks proprietary corporate knowledge. "
    "It cannot accurately answer policy-specific questions without hallucinating rules or timelines. "
    "This empirically justifies the necessity of:\n"
    "  1. Stage 3 (RAG Knowledge Base): To ground responses in authoritative corporate SOP documents.\n"
    "  2. Stage 4 (Fine-Tuned Intent Router): To accurately detect user intent and fetch the right policy."
)


=== Factual Accuracy Assessment against Corporate SOP ===

[Ground-Truth Policy (shipping_delays.md)]:
Domestic orders deliver within 3-7 business days; international orders within 10-21 business days. An order is only considered 'delayed' once it has passed the upper bound of its quoted window. Carrier scans can lag reality by up to 24 hours, so a single stale tracking event is not by itself a delay. If the parcel is late but in transit, open a carrier trace with a 24-48 hour follow-up. If no movement for >5 business days, treat as lost and offer replacement or refund.

[Baseline Model Output]:
I'm sorry to hear that your order has not arrived as expected. Please provide me with your order number so I can assist you further. As for when your order might arrive, please note that delivery times can vary depending on your location and shipping method chosen by the seller.

Regarding refunds, if there is an issue with your order, we recommend contacting the seller directly through their w

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
import json
import os

outputs_data = {
    "test_query": test_query,
    "ground_truth": ground_truth,
    "baseline_output": baseline_output
}

output_path = "outputs.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(outputs_data, f, indent=2, ensure_ascii=False)

print(f"Successfully saved baseline evaluation data to '{output_path}'.")

# Also save copies to root and Files/Notebook/ for seamless downstream execution
extra_paths = [
    os.path.join("Files", "Notebook", output_path),
    os.path.join("..", "..", output_path),
    os.path.join("..", output_path)
]
for p in extra_paths:
    parent = os.path.dirname(p)
    if parent and os.path.exists(parent):
        with open(p, "w", encoding="utf-8") as f:
            json.dump(outputs_data, f, indent=2, ensure_ascii=False)
        print(f"Saved copy to: {p}")

# Verification
with open(output_path, "r", encoding="utf-8") as f:
    reloaded_outputs = json.load(f)

print("\n--- Verification: Reloaded outputs.json ---")
for key, val in reloaded_outputs.items():
    print(f"{key}: {val[:80]}..." if len(val) > 80 else f"{key}: {val}")


Successfully saved baseline evaluation data to 'outputs.json'.
Saved copy to: ..\..\outputs.json
Saved copy to: ..\outputs.json

--- Verification: Reloaded outputs.json ---
test_query: My order hasn't arrived yet and it's been several days. When will it get here an...
ground_truth: Domestic orders deliver within 3-7 business days; international orders within 10...
baseline_output: I'm sorry to hear that your order has not arrived as expected. Please provide me...


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**